# 01. Building a Financial Support Agent Step by Step with LangGraph

In this notebook you will build a bank customer-support agent **one piece at a time**.

The core idea is to split the agent's work into two kinds:

| Kind | Examples | Owner |
|---|---|---|
| **Decisions** | Is this a legitimate inquiry? Which team owns it? Which tool? Close the case? | **Decision layer** — Bedrock Haiku 4.5 or SGLang `/v1/systemone` |
| **Generation (reasoning)** | Writing a clarifying question, fact summaries, the final customer reply | Bedrock Sonnet 4.5 (shared by both arms) |

A decision just picks one option from a set — no long text needs to be generated. The SGLang decision endpoint computes **only the label probabilities with no generation (prefill-only)**, so it finishes in tens of milliseconds and returns a confidence as well.

### Prerequisites
- An SGLang server must be running on `:8001` (Qwen3-VL-8B)
- Kernel: **Python (decision-ws)** (or the local dev kernel)


## 1. Setup: clients and the two decision backends

- `HaikuDecision`: asks Bedrock Haiku 4.5 to "answer with exactly one label as JSON"
- `SGLangDecision`: posts the question + choices to SGLang `/v1/systemone` and gets a label with confidence

Both backends receive **exactly the same question text and choices** (information parity).
> **Business context**: real contact-center organizations separate "deciding how to handle a case (rulings per the support manual and authority rules)" from "writing what to say to the customer". The decision backend owns the former; Sonnet owns the latter.


In [ ]:
import os
os.environ["DMWS_LANG"] = "en"   # localize helper output/graph captions (set BEFORE importing ws_common)
import sys; from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # ws_common path (relative inside the repo)
import ws_common as W   # graph drawing / run helpers
import sys, json, time, urllib.request
from pathlib import Path
ROOT = W.ROOT  # ws_common auto-detects the repo location

urllib.request.urlopen(f"http://{W.SERVER_HOST}:8001/health", timeout=5); print("SGLang OK")

from anthropic import AnthropicBedrock
from common import BEDROCK_MODEL_IDS, PRICING
from agent.bedrock_llm import BedrockLLM
from agent.decisions import HaikuDecision, SGLangDecision

client = AnthropicBedrock(aws_region="us-west-2", max_retries=8)
llm = BedrockLLM(client, BEDROCK_MODEL_IDS["sonnet-4-5"], PRICING["sonnet-4-5"])  # generation
haiku_dec  = HaikuDecision(client, BEDROCK_MODEL_IDS["haiku-4-5"], PRICING["haiku-4-5"])
sglang_dec = SGLangDecision(model="Qwen/Qwen3-VL-8B-Instruct")
print("clients ready")

### Try a single decision call

Before building the graph, experience what a "decision" is with one standalone call. Send the same customer message ("My wallet was stolen. Please freeze my card.") to both backends and compare the label and latency.


In [ ]:
from agent.prompts import D2_QUESTION
from agent.state import CATEGORIES

msg = "My wallet was stolen. Please freeze my card."
for name, dec in (("Haiku", haiku_dec), ("SGLang", sglang_dec)):
    r = dec.decide(state_text=msg, question=D2_QUESTION, choices=CATEGORIES, decision_id="d2")
    conf = f", confidence {r.confidence:.3f}" if r.confidence is not None else ""
    print(f"[{name:>6s}] -> {r.label:<15s} {r.latency_ms:>8,.1f}ms{conf}")

## 2. Defining the State

A LangGraph graph advances by having nodes read and update a **shared state**. Fields annotated with `Annotated[..., operator.add]` (trace, dialogue, ...) **accumulate** values returned by nodes — which is why records pile up as loops run.
> **Business context**: the State is the **support ticket (CRM case history)**. Customer utterances, intermediate rulings, system lookups and handling memos must accumulate on one ticket for later complaint handling and audits — that is what `trace` does.


In [ ]:
import operator
from typing import Annotated, List, Optional, TypedDict

class AgentState(TypedDict, total=False):
    conv_id: str; arm: str; lang: str
    script: dict                                   # scripted tool outcomes / customer replies
    dialogue: Annotated[List[dict], operator.add]  # accumulated customer/agent turns
    trace: Annotated[List[dict], operator.add]     # accumulated observability events
    d1: str; d2: str; d4: str; d7: str; d8: str
    d5_labels: Annotated[List[str], operator.add]
    d3_labels: Annotated[List[str], operator.add]
    d6_labels: Annotated[List[str], operator.add]
    tools_used: Annotated[List[str], operator.add]
    clarify_asks: int; tool_attempts: int
    tool_result: Optional[dict]; gave_up: bool
    kyc_result: Optional[dict]
    specialist_summary: str; final_response: str

print("state defined")

## 3. Observability helpers and constants

Every node records what it did (decision label/latency, generation latency, tool results) into `trace`. The scenario notebooks later render this trace step by step. Recursive loops get **hard caps** (no infinite loops).
> **Business context**: audit trails are mandatory in financial support automation. You must be able to show which ruling was made by which model, in how many ms, on what grounds (confidence) for internal control and examinations. The hard caps are the coded form of rules like "no indefinite holds".


In [ ]:
MAX_CLARIFY_ASKS = 2    # clarification loop cap
MAX_TOOL_ATTEMPTS = 2   # tool retry loop cap

def conv_text(state):
    return "\n".join(f"{t['role']}: {t['text']}" for t in state["dialogue"])

def dec_event(node, decision_id, dr, backend):
    return {"node": node, "kind": "decision", "decision": decision_id, "backend": backend,
            "label": dr.label, "confidence": dr.confidence, "valid": dr.valid,
            "latency_ms": dr.latency_ms, "usage": dr.usage, "cost_usd": dr.cost_usd}

def llm_event(node, lr, backend):
    return {"node": node, "kind": "llm", "backend": backend,
            "latency_ms": lr.latency_ms, "usage": lr.usage, "cost_usd": lr.cost_usd}

def tool_event(node, t0):
    return {"node": node, "kind": "tool",
            "latency_ms": round((time.perf_counter() - t0) * 1000.0, 3)}

decision = sglang_dec   # assemble with the SGLang backend here (swap freely)
print("helpers ready")

## 4. Nodes 1-2: intake screening (D1) and routing (D2)

- **guardrail_check [D1]**: legitimate inquiry (in_scope) / unrelated to banking (out_of_scope) / compliance violation (sensitive_compliance). Abnormal inquiries skip the specialists and go straight to the reply.
- **intent_router [D2]**: pick one of 12 specialist teams — the contact center's skill-group routing.
> **Business context**: D1 is the **first screening at intake** — separating out-of-scope calls and refusing AML/fraud-assistance requests. D2 corresponds to **skill-group routing** (IVR branching; assigning cards/deposits/FX specialist teams).


In [ ]:
from agent.prompts import D1_QUESTION, D2_QUESTION, D3_QUESTION, D4_QUESTION
from agent.prompts import RESPOND_SYSTEM, SPECIALIST_SYSTEM
from agent.prompts_v2 import (ASK_CLARIFY_SYSTEM, D5_CHOICES, D5_QUESTION,
                              D6_CHOICES, D6_QUESTION, D7_CHOICES, D7_QUESTION,
                              D8_CHOICES, D8_QUESTION)
from agent.state import CATEGORIES, D1_CHOICES, D4_CHOICES
from agent.tools_v2 import CATEGORY_TOOLS, run_kyc, run_tool_v2

def guardrail_check(state: AgentState):
    dr = decision.decide(state_text=state["dialogue"][0]["text"], question=D1_QUESTION,
                         choices=D1_CHOICES, decision_id="d1")
    return {"d1": dr.label, "trace": [dec_event("guardrail_check", "d1", dr, decision.name)]}

def intent_router(state: AgentState):
    dr = decision.decide(state_text=state["dialogue"][0]["text"], question=D2_QUESTION,
                         choices=CATEGORIES, decision_id="d2")
    return {"d2": dr.label, "trace": [dec_event("intent_router", "d2", dr, decision.name)]}

print("nodes: guardrail_check, intent_router")

**The graph so far** — the skeleton: a legitimate inquiry flows from intake screening (D1) to routing (D2); an abnormal one takes the dashed path straight to the final (refusal) reply. Green borders mark nodes added in this step.


In [ ]:
W.draw_build_stage(1)

## 5. Recursive loop 1 — clarification

When the inquiry is vague, the agent **asks back**, receives the answer, then **re-decides**.

```
clarify_gate [D5] --need_clarification--> ask_clarify --(loop)--> clarify_gate
             └--info_sufficient / budget exhausted--> tool_step
```

- `clarify_gate` decides D5 over the **entire conversation** (the conversation grows as the loop runs)
- In `ask_clarify`, Sonnet writes the question; the customer reply is scripted for reproducibility
- When the cap (2 asks) is reached we proceed anyway — the loop safety valve
> **Business context**: this is the **information completion procedure** where an agent asks "which card, which transaction?". Dispute filings and incident reports cannot be accepted without identifying the transaction (date, amount, merchant), so this back-and-forth is mandatory in real operations too.


In [ ]:
def clarify_gate(state: AgentState):
    dr = decision.decide(state_text="Conversation so far:\n" + conv_text(state),
                         question=D5_QUESTION, choices=D5_CHOICES, decision_id="d5")
    return {"d5_labels": [dr.label], "trace": [dec_event("clarify_gate", "d5", dr, decision.name)]}

def route_clarify(state: AgentState):
    if (state["d5_labels"][-1] == "need_clarification"
            and state.get("clarify_asks", 0) < MAX_CLARIFY_ASKS):
        return "ask_clarify"
    return "tool_step"

def ask_clarify(state: AgentState):
    lr = llm.generate(system=ASK_CLARIFY_SYSTEM,
                      user="Conversation so far:\n" + conv_text(state), max_tokens=120)
    asks = state.get("clarify_asks", 0)
    s = state["script"]
    reply = s["clarify_reply"] if (s.get("vague") and asks == 0) else s["no_info_reply"]
    return {"clarify_asks": asks + 1,
            "dialogue": [{"role": "agent", "text": lr.text},
                         {"role": "customer", "text": reply}],
            "trace": [llm_event("ask_clarify", lr, llm.name)]}

print("nodes: clarify_gate, ask_clarify (+route_clarify)")

**The graph so far** — the **first recursive loop** (red edges) appears after routing. If information is missing we ask back, take the reply, and re-decide.


In [ ]:
W.draw_build_stage(2)

## 6. Recursive loop 2 — tool selection, execution and retry

A single `tool_step` does three things: **D3** pick a tool from the team's tools (excluding ones already tried) → run the tool → **D6** review the result.

```
tool_step [D3+tool+D6] --try_alternative--> tool_step   (retry with another tool)
         ├--goal_met--> summarize
         └--give_up--> escalation_gate                   (human handoff path)
```

In this workshop the tool returns success / tool_error / blocked per the script — so the correct D6 answer is objective by construction.
> **Business context**: tool calls correspond to **core-banking / card-ledger / external-system transactions**. Transient failures are routine in production systems, so operations have a procedure of "read the error code, retry with an alternative transaction, hand off to manual processing if it still fails" — the D6 loop reproduces exactly that.


In [ ]:
def tool_step(state: AgentState):
    category = state["d2"]
    attempt = state.get("tool_attempts", 0)
    used = state.get("tools_used", [])
    choices = [t for t in CATEGORY_TOOLS[category] if t not in used]
    events = []
    if len(choices) == 1:   # single remaining candidate: auto-select (no decision needed)
        tool = choices[0]
        events.append({"node": "tool_select", "kind": "auto", "decision": "d3",
                       "label": tool, "latency_ms": 0.0, "cost_usd": 0.0})
    else:
        dr = decision.decide(
            state_text="Conversation so far:\n" + conv_text(state)
                       + (f"\n\nAlready tried (failed): {', '.join(used)}" if used else ""),
            question=D3_QUESTION.format(category=category), choices=choices, decision_id="d3")
        tool = dr.label
        events.append(dec_event("tool_select", "d3", dr, decision.name))
    t0 = time.perf_counter()
    tool_result = run_tool_v2(tool, state["conv_id"], state["script"], attempt)
    events.append(tool_event(f"tool:{tool}", t0))
    dr6 = decision.decide(
        state_text=(f"Customer request:\n{state['dialogue'][0]['text']}\n\n"
                    f"Tool executed: {tool}\nTool result:\n"
                    f"{json.dumps(tool_result, ensure_ascii=False)}"),
        question=D6_QUESTION, choices=D6_CHOICES, decision_id="d6")
    events.append(dec_event("tool_review", "d6", dr6, decision.name))
    return {"d3_labels": [tool], "d6_labels": [dr6.label], "tools_used": [tool],
            "tool_attempts": attempt + 1, "tool_result": tool_result, "trace": events}

def route_tool(state: AgentState):
    label = state["d6_labels"][-1]
    remaining = [t for t in CATEGORY_TOOLS[state["d2"]]
                 if t not in state.get("tools_used", [])]
    if (label == "try_alternative"
            and state.get("tool_attempts", 0) < MAX_TOOL_ATTEMPTS and remaining):
        return "tool_step"
    if label == "goal_met":
        return "summarize"
    return "give_up"

def mark_give_up(state: AgentState):
    return {"gave_up": True}

print("nodes: tool_step, give_up (+route_tool)")

**The graph so far** — the **second recursive loop**: when the tool-result review (D6) says `try_alternative`, the node loops back to itself to retry with another tool. A `give_up` path for abandoning automated handling has appeared as well.


In [ ]:
W.draw_build_stage(3)

## 7. The back half: summary, risk check (D7), KYC (D8), final action (D4), reply

- **summarize**: Sonnet condenses the tool result into customer-facing facts (generation)
- **risk_check [D7]**: if the tool result carries a fraud/AML monitoring flag, branch to KYC re-verification
- **kyc_verify**: run the KYC tool + **D8** pass/fail decision
- **escalation_gate [D4]**: resolve / clarify / human_handoff
- **respond**: Sonnet writes the final reply following the decided action (generation)
> **Business context**: summarize is the **case-handling memo** (history for follow-up agents and complaint handling); D7 is checking the **FDS (fraud detection) alert**; kyc_verify is **customer re-verification (KYC/CDD)** — the coded form of "never close a high-risk transaction without re-verification". D4 is the **authority decision** (auto-closable vs beyond authority, hand off), and respond is **customer communication** in complaint-preventing standard language.


In [ ]:
def summarize(state: AgentState):
    lr = llm.generate(
        system=SPECIALIST_SYSTEM.format(category=state["d2"]),
        user=(f"Conversation:\n{conv_text(state)}\n\n"
              f"Tool executed: {state['tool_result']['tool']}\nTool result:\n"
              f"{json.dumps(state['tool_result'], ensure_ascii=False)}"), max_tokens=300)
    return {"specialist_summary": lr.text, "trace": [llm_event("summarize", lr, llm.name)]}

def risk_check(state: AgentState):
    ctx = (f"Case facts:\n{state.get('specialist_summary', '')}\n\n"
           f"Tool result:\n{json.dumps(state.get('tool_result'), ensure_ascii=False)}")
    dr = decision.decide(state_text=ctx, question=D7_QUESTION,
                         choices=D7_CHOICES, decision_id="d7")
    return {"d7": dr.label, "trace": [dec_event("risk_check", "d7", dr, decision.name)]}

def kyc_verify(state: AgentState):
    t0 = time.perf_counter()
    kyc = run_kyc(state["conv_id"], state["script"])
    events = [tool_event("tool:run_kyc_check", t0)]
    dr = decision.decide(state_text=f"KYC result:\n{json.dumps(kyc, ensure_ascii=False)}",
                         question=D8_QUESTION, choices=D8_CHOICES, decision_id="d8")
    events.append(dec_event("kyc_review", "d8", dr, decision.name))
    return {"d8": dr.label, "kyc_result": kyc, "trace": events}

def escalation_gate(state: AgentState):
    parts = [f"Conversation:\n{conv_text(state)}",
             f"Tool result:\n{json.dumps(state.get('tool_result'), ensure_ascii=False)}"]
    if state.get("specialist_summary"):
        parts.append(f"Specialist facts:\n{state['specialist_summary']}")
    if state.get("kyc_result") is not None:
        parts.append(f"KYC result:\n{json.dumps(state['kyc_result'], ensure_ascii=False)}")
    dr = decision.decide(state_text="\n\n".join(parts), question=D4_QUESTION,
                         choices=D4_CHOICES, decision_id="d4")
    return {"d4": dr.label, "trace": [dec_event("escalation_gate", "d4", dr, decision.name)]}

def respond(state: AgentState):
    action = state["d1"] if state["d1"] != "in_scope" else state.get("d4", "resolve")
    parts = [f"Decided action: {action}", f"Conversation:\n{conv_text(state)}"]
    if state.get("specialist_summary"):
        parts.append(f"Resolution facts:\n{state['specialist_summary']}")
    if state.get("tool_result"):
        parts.append(f"Tool result:\n{json.dumps(state['tool_result'], ensure_ascii=False)}")
    if state.get("kyc_result") is not None:
        parts.append(f"KYC result:\n{json.dumps(state['kyc_result'], ensure_ascii=False)}")
    lr = llm.generate(system=RESPOND_SYSTEM, user="\n\n".join(parts), max_tokens=400)
    return {"final_response": lr.text, "trace": [llm_event("respond", lr, llm.name)]}

print("nodes: summarize, risk_check, kyc_verify, escalation_gate, respond")

**The graph so far** — summary, risk check (D7) and the conditional KYC sub-flow (D8) are in. Only the final action (D4) and the reply remain to be wired.


In [ ]:
W.draw_build_stage(4)

## 8. Assembling the graph

Register the nodes and connect the edges. **Conditional edges** create the branches and the recursive loops.
> **Business context**: this wiring is the step that turns the **process flowchart (BPM) from the operations manual into executable code**. The conditional edges are the manual's branching rules ("refuse sensitive inquiries immediately", "re-verify on an FDS alert").


In [ ]:
from langgraph.graph import StateGraph, START, END

g = StateGraph(AgentState)
for name, fn in [("guardrail_check", guardrail_check), ("intent_router", intent_router),
                 ("clarify_gate", clarify_gate), ("ask_clarify", ask_clarify),
                 ("tool_step", tool_step), ("give_up", mark_give_up),
                 ("summarize", summarize), ("risk_check", risk_check),
                 ("kyc_verify", kyc_verify), ("escalation_gate", escalation_gate),
                 ("respond", respond)]:
    g.add_node(name, fn)

g.add_edge(START, "guardrail_check")
g.add_conditional_edges("guardrail_check",
    lambda s: "intent_router" if s["d1"] == "in_scope" else "respond",
    ["intent_router", "respond"])                       # abnormal inquiries go straight to the reply
g.add_edge("intent_router", "clarify_gate")
g.add_conditional_edges("clarify_gate", route_clarify, ["ask_clarify", "tool_step"])
g.add_edge("ask_clarify", "clarify_gate")               # <= recursive loop 1
g.add_conditional_edges("tool_step", route_tool, ["tool_step", "summarize", "give_up"])  # <= loop 2
g.add_edge("give_up", "escalation_gate")
g.add_edge("summarize", "risk_check")
g.add_conditional_edges("risk_check",
    lambda s: "kyc_verify" if s["d7"] == "needs_verification" else "escalation_gate",
    ["kyc_verify", "escalation_gate"])
g.add_edge("kyc_verify", "escalation_gate")
g.add_edge("escalation_gate", "respond")
g.add_edge("respond", END)

my_graph = g.compile()
print("compiled. nodes:", list(my_graph.get_graph().nodes))

In [ ]:
# the completed graph (localized captions, red edges = recursive loops)
W.draw_build_stage(5)

## 9. Test drive

Run one simple inquiry through the graph you just assembled. `ws_common.run_verbose()` streams node execution and prints **which decision/generation happens at each STEP**. Test utterance: "I received my new debit card and would like to activate it. Card ending 7712."


In [ ]:

res = W.run_verbose(my_graph,
    "I received my new debit card and would like to activate it. Card ending 7712.",
    {"vague": False, "clarify_reply": None, "no_info_reply": W.NO_INFO_EN,
     "tool_outcomes": ["success"], "risk_flag": False, "kyc_result": None},
    "SGLang (hand-built)")


## 10. Wrap-up

- The agent = **8 decision points (D1-D8)** + **3 generation nodes** + 2 recursive loops + a conditional KYC sub-flow.
- The code you wrote here is identical to `build_graph_v2(decision, llm)` in `agent/graph_v2.py`. From the next notebooks ([EN]_02-04) we import it to build the **Haiku arm and the SGLang arm with the same topology** and compare them on support scenarios.
- Set `decision = haiku_dec` and re-run this notebook: the same graph now runs on Haiku decisions — try the comparison yourself.
